In [30]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import numpy as np
import pandas as pd
import os 
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, OPTICS, AffinityPropagation 
from sklearn.cluster import estimate_bandwidth # for MeanShift clustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder,OneHotEncoder, LabelEncoder # preprocessing
from sklearn.metrics import silhouette_score, adjusted_rand_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer # transform columns
import time # to calculate execution time
from scipy.optimize import linear_sum_assignment # match cluster labels to true class labels optimally
from joblib import Parallel, delayed
import itertools # to create combinations of hyperparameters
from sklearn.base import BaseEstimator, TransformerMixin, ClusterMixin # for Affinity Propagation and MeanShift clustering
from sklearn.metrics import pairwise_distances # for Affinity Propagation

import plotly.express as px # for visualization




### Openml
In Python, OpenML is mainly used to discover, download, and share ML datasets, tasks, and results—super handy for experiments, benchmarking, and learning ML properly.

In [32]:
%pip install openml




[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import openml

### Download the dataset from openl using dataset id

In [34]:
def download_dataset(dataset_id):
    dataset = openml.datasets.get_dataset(dataset_id)
    X,y, categorical_indicator, attribute_names = dataset.get_data(
    target=dataset.default_target_attribute)
    return X,y, categorical_indicator, attribute_names


### Prepare data
Define numerical and categorical columns based on the categorical_indicator

In [35]:
# Define columns types
def define_column_types(X):
    cat_columns = X.columns[np.array(categorical_indicator)==True] # cat column names
    num_columns = X.columns[np.array(categorical_indicator)==False] # numerical column names
    return list(num_columns), list(cat_columns)

# Pre-processing
Numeric variables --> scale
Ordinal variables -->  Ordinal encoding
Nominal categorical variables -->  Onehot encoding
Binary(already 0 and 1) -->  onehot encoding 


# Define parameters

In [36]:
def define_parameters(algorithm):
    if algorithm == "KMeans":
        param_grid = {
        "n_clusters": [2, 3, 4, 5, 6],
        "init": ["k-means++", "random"],
        "n_init": [10, 20],
        "max_iter": [300, 500]
        }

    if algorithm == "AgglomerativeClustering":
        param_grid ={
        "n_clusters": [2, 3, 4,5,6],
        "linkage": ["single","average", "complete"],
        "metric": ["euclidean", "manhattan"] 
        }

    if algorithm == "Ward":
        param_grid ={
        "n_clusters": [2, 3, 4,5,6],
        "linkage": ["ward"], # only ward can be used
        "metric": ["euclidean"] # only euclidean can be used
        }
    
    if algorithm == "DBSCAN":
        param_grid = {
            "eps": [0.1,0.5, 0.7,0.8],
            "min_samples": [10, 20, 30],
            "metric": ["euclidean"],
            "algorithm":["auto", "ball_tree", "kd_tree", "brute"]
        }

    if algorithm == "OPTICS":
        param_grid={
            "min_samples": [10,20,30],
            "max_eps": [0.5],
            "xi": [0.05, 0.1],
            "min_cluster_size": [10, 20],
            "metric": ["euclidean"],
            "algorithm":["auto", "ball_tree", "kd_tree", "brute"]
        }
    
    if algorithm == "GaussianMixture":
        param_grid = {
            "n_components": [2,3,4], # clusters
            "covariance_type": ["full", "tied"],
            "init_params" : ['kmeans', 'random']
        }

    if algorithm == "AffinityPropagation":
        param_grid = {
            "damping": [0.5, 0.7], # Damping factor to stabilize updates (0.5–1.0). Prevents oscillations.
            "max_iter": [300, 500], # Maximum number of iterations
            "convergence_iter": [10,20], # Number of iterations with no change in cluster assignments to declare convergence
             "affinity": ['euclidean','precomputed'] # Number of iterations with no change in cluster assignments to declare convergence
        }

    if algorithm == "MeanShift":
        param_grid = {
            "quantile": [0.1, 0.2, 0.3],
            "bin_seeding": [True, False],
            "n_samples": [30,40] 
        }
    
        
    param_names = list(param_grid.keys())
    param_combinations =list(itertools.product(    
        *(param_grid[param_name] for param_name in param_names))
        )
    return  param_grid, param_combinations, param_names


### Define similarity matrix for Affinity Propagation

In [37]:
class SimilarityTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, metric='euclidean'):
        self.metric = metric  # The distance metric to use for computing similarities
    
    def fit(self, X, y=None):
        # Nothing to learn here; just return self
        return self
    
    def transform(self, X):
        # Compute negative pairwise distances as a similarity matrix
        similarity_matrix = -pairwise_distances(X, metric=self.metric)
        return similarity_matrix

### create a custom andwidth estimator for MeanShift clustering

In [38]:
class MeanShiftAutoBW(BaseEstimator, ClusterMixin):
    """
    Mean-Shift clustering wrapper for pipelines.
    Allows bandwidth estimation from a quantile.
    """
    def __init__(self, quantile=0.2, n_samples=400, bin_seeding=True):
        self.quantile = quantile
        self.n_samples = n_samples
        self.bin_seeding = bin_seeding

    def fit(self, X, y=None):
        # Estimate bandwidth from data
        self.bandwidth_ = estimate_bandwidth(
            X,
            quantile=self.quantile,
            n_samples=self.n_samples
        )

        # Fit Mean-Shift with the computed bandwidth
        self.model_ = MeanShift(
            bandwidth=self.bandwidth_,
            bin_seeding=self.bin_seeding
        )
        self.model_.fit(X)

        # Store labels
        self.labels_ = self.model_.labels_
        return self

    def predict(self, X):
        """
        Mean-Shift does not support true prediction.
        We return labels for the fitted data only.
        """
        return self.labels_

# Define model

In [39]:
def evaluate_performance(params):
    start = time.time()

    keys = param_names
    params_dict = dict(zip(keys, params))
    #print(params_dict)

    if algorithm in ["KMeans", "GaussianMixture"]:
        pipe = Pipeline([
            ("preprocess" , preprocessor),
            (algorithm, algorithms[algorithm](**params_dict, random_state=42))
            ])
        y_pred = pipe.fit_predict(X)

    if algorithm == "AffinityPropagation":
         pipe = Pipeline([
            ("preprocess" , preprocessor),
            ("similarity", SimilarityTransformer(metric='euclidean')),
            (algorithm, algorithms[algorithm](**params_dict, random_state=42))
            ])
         y_pred = pipe.fit_predict(X)
    else:
        pipe = Pipeline([
            ("preprocess" , preprocessor),
            (algorithm, algorithms[algorithm](**params_dict))
            ])
        y_pred = pipe.fit_predict(X)
    
    #y_pred = pipe.fit_predict(X)
   
    # Access the last step (KMeans model)
    # last_step = pipe.steps[-1][1]
    # y_pred = last_step.predict(X) 
    # if hasattr(last_step, "labels_"):
    #     labels = last_step.labels_
    # else:
    #     labels = last_step.predict(X)
    
    #y_pred = pipe.fit_predict(X)

    # Filter the predicted noise cluster(-1) from labels and features
    # mask = y_predicted !=1 # create a mask to remove clusters = -1 which is the noise detected by DBSCAN algorithm
    # y_pred =y_predicted[mask] # filter the noise from labels
    # X_clustered = X[mask] # filter the noise from features
   
    silhouette = silhouette_score(X, y_pred)

    # # silhouette_score
    # n_clusters = np.unique(y_pred)
    # if n_clusters >= 2:
    #     silhouette = silhouette_score(X, y_pred)
    # else:
    #     silhouette = None
    
    # adjusted_rand_score
    ari = adjusted_rand_score(y_enc, y_pred) # no need to do label alignment

    # f1_score
    # Align cluster labels to true labels using Hungarian algorithm
    def align_cluster_labels(y_enc, y_pred):
        cm = confusion_matrix(y_enc, y_pred)  
        row_ind, col_ind = linear_sum_assignment(-cm) # # Maximize diagonal → minimize negative
        mapping = {col: row for row, col in zip(row_ind, col_ind)} 
        return np.array([mapping[label] for label in y_pred])

    y_pred_aligned = align_cluster_labels(y_enc, y_pred)
    f1 = f1_score(y_enc, y_pred_aligned, average="macro") # Computes F1 per class, takes an unweighted mean, treats all clusters/classes equally
    
   
    execution_time = float(time.time() - start)
    #params_dict["inertia"] = pipe.named_steps[algorithm].inertia_
    params_dict["silhouette_score"] = silhouette
    params_dict["adjusted_rand_score"] = ari
    params_dict["f1_score"] = f1
    params_dict["execution_time"] = execution_time
    

    return params_dict



### Joblib 
joblib is a Python library mainly used in ML for saving models, fast loading, and parallel processing

n_jobs answers “how many things can run in parallel? n_jobs does NOT say threads or processes.
Threading is how parallelism is done. Threading is a backend choice in joblib.
backend="threading"   # threads
backend="loky"        # processes (default)

### Create lists of openml datasets and datasets to be analyzed

In [ ]:
dataset_names  = ["iris", "wine","haberman", "libras_move"]
#"glass" - exception in cat_indicator #, , , "isolet",, "gas-drift-different-concentrations", "MagicTelescope","letter", "covertype"] # datasets to be analyzed
# "satellite_image" - memory issue
# "isolet" - too large
# "nursery" - could not convert from string to float
datasets = openml.datasets.list_datasets(output_format="dataframe") # openml datasets

In [41]:
algorithms = {
    "KMeans": KMeans, 
    "AgglomerativeClustering": AgglomerativeClustering,
    "Ward": AgglomerativeClustering,
    #"DBSCAN": DBSCAN,
    #"OPTICS": OPTICS,
    "GaussianMixture": GaussianMixture,
    "AffinityPropagation": AffinityPropagation,
    #"Meanshift": MeanShiftAutoBW
}

results_all = pd.DataFrame()

for algorithm in algorithms:
    for dataset_name in dataset_names:
        dataset_id = int(datasets.loc[datasets["name"] == dataset_name, "did"].values[0])
        # Download dataset
        X,y, categorical_indicator, attribute_names = download_dataset(dataset_id)
        
        # define trypes of columns in X
        num_columns, cat_columns = define_column_types(X) 

        X[num_columns]= X[num_columns].astype("float32") # to reduce memory due to float64

        # preprocess y
        label_enc = LabelEncoder()
        y_enc = label_enc.fit_transform(y)

        # define preprocessor for X
        try:
            ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

        preprocessor = ColumnTransformer(
            transformers = [
                ("num", MinMaxScaler(), num_columns),
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns),
            ],
            sparse_threshold=0,
        )

        
        # define parameter grid
        param_grid, param_combinations, param_names = define_parameters(algorithm)
    
        # evaluate performance of each algorithm using parallel processing

        # results = Parallel(n_jobs=-1, verbose=10)(
        # delayed(evaluate_performance)(params) for params in param_combinations
        # )

        # Number of parallel jobs
        n_workers = min(4, max(1, os.cpu_count() - 1)) # leave one core available for other tasks, such as operating system processes
        results = Parallel(n_jobs=n_workers, verbose=10, backend="threading", pre_dispatch='n_jobs')(
            delayed(evaluate_performance)(params) for params in param_combinations
            )

        results_df = pd.DataFrame(results)
        results_df.insert(0,"dataset", dataset_name)
        results_df.insert(1,"algorithm", algorithm)
        results_all = pd.concat([results_all, results_df], ignore_index=True)
        print(results_all[results_all["dataset"]=="wine"])
        
results_all.to_csv("results.csv")

            
    
   

 


    


    



[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:


Empty DataFrame
Columns: [dataset, algorithm, n_clusters, init, n_init, max_iter, silhouette_score, adjusted_rand_score, f1_score, execution_time]
Index: []


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:


   dataset algorithm  n_clusters       init  n_init  max_iter  \
40    wine    KMeans           2  k-means++      10       300   
41    wine    KMeans           2  k-means++      10       500   
42    wine    KMeans           2  k-means++      20       300   
43    wine    KMeans           2  k-means++      20       500   
44    wine    KMeans           2     random      10       300   
45    wine    KMeans           2     random      10       500   
46    wine    KMeans           2     random      20       300   
47    wine    KMeans           2     random      20       500   
48    wine    KMeans           3  k-means++      10       300   
49    wine    KMeans           3  k-means++      10       500   
50    wine    KMeans           3  k-means++      20       300   
51    wine    KMeans           3  k-means++      20       500   
52    wine    KMeans           3     random      10       300   
53    wine    KMeans           3     random      10       500   
54    wine    KMeans     

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.

c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.

[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.1s
c:\Users\Sajindra\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to

...

    dataset            algorithm  n_clusters       init  n_init  max_iter  \
40     wine               KMeans         2.0  k-means++    10.0     300.0   
41     wine               KMeans         2.0  k-means++    10.0     500.0   
42     wine               KMeans         2.0  k-means++    20.0     300.0   
43     wine               KMeans         2.0  k-means++    20.0     500.0   
44     wine               KMeans         2.0     random    10.0     300.0   
..      ...                  ...         ...        ...     ...       ...   
462    wine  AffinityPropagation         NaN        NaN     NaN     300.0   
463    wine  AffinityPropagation         NaN        NaN     NaN     500.0   
464    wine  AffinityPropagation         NaN        NaN     NaN     500.0   
465    wine  AffinityPropagation         NaN        NaN     NaN     500.0   
466    wine  AffinityPropagation         NaN        NaN     NaN     500.0   

     silhouette_score  adjusted_rand_score  f1_score  execution_time linkag

[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done   4 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done   9 tasks      | elapsed:    0.9s
[Parallel(n_jobs=4)]: Done  16 out of  16 | elapsed:    1.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


    dataset            algorithm  n_clusters       init  n_init  max_iter  \
40     wine               KMeans         2.0  k-means++    10.0     300.0   
41     wine               KMeans         2.0  k-means++    10.0     500.0   
42     wine               KMeans         2.0  k-means++    20.0     300.0   
43     wine               KMeans         2.0  k-means++    20.0     500.0   
44     wine               KMeans         2.0     random    10.0     300.0   
..      ...                  ...         ...        ...     ...       ...   
462    wine  AffinityPropagation         NaN        NaN     NaN     300.0   
463    wine  AffinityPropagation         NaN        NaN     NaN     500.0   
464    wine  AffinityPropagation         NaN        NaN     NaN     500.0   
465    wine  AffinityPropagation         NaN        NaN     NaN     500.0   
466    wine  AffinityPropagation         NaN        NaN     NaN     500.0   

     silhouette_score  adjusted_rand_score  f1_score  execution_time linkag

[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:  1.5min
[Parallel(n_jobs=4)]: Done   4 tasks      | elapsed:  9.0min
[Parallel(n_jobs=4)]: Done   9 tasks      | elapsed: 12.8min


KeyboardInterrupt: 

### Visualize the results

1. for each dataset compare and contrast results produced by each algorithm under optimal parameter settings

In [ ]:
# For each dataset and algorithm Maximum f1 score
max_f1 = results_all.loc[results_all.groupby(["dataset", "algorithm"])["f1_score"].idxmax()]
#print(max_f1)

fig = px.bar(
    max_f1,
    x = "dataset",
    y = "f1_score",
    color = "algorithm",
    barmode="group",
    title="Maximum F1 Score for Each Clustering  Datasets Across Algorithms",
    labels={"f1_score": "F1 Score", "algorithm": "Clustering Algorithm"},
)
fig.show()

In [ ]:
# For each dataset and algorithm maximum silhouette_score
max_f1 = results_all.loc[results_all.groupby(["dataset", "algorithm"])["silhouette_score"].idxmax()]
#print(max_f1)

fig = px.bar(
    max_f1,
    x = "dataset",
    y = "silhouette_score",
    color = "algorithm",
    barmode="group",
    title="Maximum silhouette_score for Each Datasets Across Clustering Algorithms",
    labels={"silhouette_score": "silhouette_score", "algorithm": "Clustering Algorithm"},
)
fig.show()

In [ ]:
# For each dataset and algorithm maximum adjusted_rand_score
max_f1 = results_all.loc[results_all.groupby(["dataset", "algorithm"])["adjusted_rand_score"].idxmax()]
#print(max_f1)

fig = px.bar(
    max_f1,
    x = "dataset",
    y = "adjusted_rand_score",
    color = "algorithm",
    barmode="group",
    title="Maximum adjusted_rand_score for Each Datasets Across Clustering Algorithms",
    labels={"adjusted_rand_score": "adjusted_rand_score", "algorithm": "Clustering Algorithm"},
)
fig.show()

2. for each given algorithm, how f1_score, adjusted_rand_score and silhouette_score varies with the different parameter values

In [ ]:
# AffinityPropagation - f1_score Vs parameters 
param_grid, param_combinations, param_names = define_parameters("AffinityPropagation")

affinityPropagation_df = results_all.loc[results_all["algorithm"]=="AffinityPropagation"]

for param_name in param_names:
    fig = px.box(
    affinityPropagation_df,
    x = param_name,
    y = "f1_score",
    title= f"AffinityPropagation - f1_score Vs {param_name}")
    fig.show()



In [67]:
# AffinityPropagation - silhouette_score_score Vs parameters 

for param_name in param_names:
    fig = px.box(
    affinityPropagation_df,
    x = param_name,
    y = "silhouette_score",
    title= f"AffinityPropagation - silhouette_score Vs {param_name}")
    fig.show()

In [68]:
# AffinityPropagation - adjusted_rand_score Vs parameters 

for param_name in param_names:
    fig = px.box(
    affinityPropagation_df,
    x = param_name,
    y = "adjusted_rand_score",
    title= f"AffinityPropagation - adjusted_rand_score Vs {param_name}")
    fig.show()